<a href="https://colab.research.google.com/github/honourjesus/Bilingual-Hausa-English-ASR-TTS-System/blob/main/WAECHAUNMTwithine_tuned_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1: Install dependencies

!pip install transformers sentencepiece sacrebleu -q

In [ ]:
# CELL 2: Load JSON and flatten to DataFrame

import json
import pandas as pd

with open("/content/questions_merged.json", "r") as f:  # adjust path if needed
    data = json.load(f)

rows = []
for subject, years in data.items():
    for year, langs in years.items():
        for q in langs.get("English", []):
            rows.append({
                "subject": subject,
                "year": year,
                "question_number": q["question_number"],
                "question_text_en": q["question_text"],
                "correct_answer_en": q["correct_answer"],
            })

df = pd.DataFrame(rows)
print(f" Total questions: {len(df)}")
print(df.head(3))

 Total questions: 5648
            subject  year question_number  \
0  Animal Husbandry  2017               1   
1  Animal Husbandry  2017               2   
2  Animal Husbandry  2017               3   

                                    question_text_en      correct_answer_en  
0  The number of teats in a cow is A. two B. four...                B. four  
1  The movement of digested feed into the bloodst...          A. absorption  
2  Selection of animals based on performance of a...  D. pedigree selection  


In [ ]:
# CELL 3: Load your fine-tuned model from Hugging Face

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "honourjesus/nllb-hausa-waec"  # ← your HF repo

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"✅ Model loaded on {device}")

config.json:   0%|          | 0.00/845 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/31.9M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/5.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

✅ Model loaded on cuda


In [ ]:
# CELL 4: Translation function

def translate_batch(texts, batch_size=16):
    translations = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids("hau_Latn"),
                max_length=256,
                num_beams=4,
            )
        translations.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
        print(f"  Translated {min(i+batch_size, len(texts))}/{len(texts)}", end="\r")
    return translations

In [ ]:
# CELL 5: Translate questions and answers

print("Translating questions...")
df["question_text_ha_pred"] = translate_batch(df["question_text_en"].tolist())

print("\nTranslating answers...")
df["correct_answer_ha_pred"] = translate_batch(df["correct_answer_en"].tolist())

print("\n✅ Done! Sample output:")
df[["subject", "year", "question_text_en", "question_text_ha_pred",
    "correct_answer_en", "correct_answer_ha_pred"]].head(3)

Translating questions...
  Translated 5648/5648
Translating answers...
  Translated 5648/5648
✅ Done! Sample output:


,subject,year,question_text_en,question_text_ha_pred,correct_answer_en,correct_answer_ha_pred
0,Animal Husbandry,2017,The number of teats in a cow is A. two B. four...,Adadin nono a cikin saniya shine Option A. biy...,B. four,B. Hudu
1,Animal Husbandry,2017,The movement of digested feed into the bloodst...,Motsa abincin da aka narke cikin jini ana kira...,A. absorption,Option A. Absorption
2,Animal Husbandry,2017,Selection of animals based on performance of a...,Zaɓin dabbobi bisa ga aikin kakanninsu ana kir...,D. pedigree selection,Option D. Zaɓin asalin


In [ ]:
df[["subject", "year", "question_text_en", "question_text_ha_pred",
    "correct_answer_en", "correct_answer_ha_pred"]].head(50)

,subject,year,question_text_en,question_text_ha_pred,correct_answer_en,correct_answer_ha_pred
0,Animal Husbandry,2017,The number of teats in a cow is A. two B. four...,Adadin nono a cikin saniya shine Option A. biy...,B. four,B. Hudu
1,Animal Husbandry,2017,The movement of digested feed into the bloodst...,Motsa abincin da aka narke cikin jini ana kira...,A. absorption,Option A. Absorption
2,Animal Husbandry,2017,Selection of animals based on performance of a...,Zaɓin dabbobi bisa ga aikin kakanninsu ana kir...,D. pedigree selection,Option D. Zaɓin asalin
3,Animal Husbandry,2017,The incubation period of chicken egg is A. sev...,Lokacin injin kaza shine Option A. kwana goma ...,B. twenty-one days,B. kwanaki ashirin da ɗaya
4,Animal Husbandry,2017,Which of the following body systems determines...,Wanne daga cikin waɗannan tsarin jikin yake ta...,B. Digestive,B. Digestive
5,Animal Husbandry,2017,Which of the following farm animals are monoga...,Wanne daga cikin waɗannan dabbobin gona ne suk...,C. II and III,Option C. II da III
6,Animal Husbandry,2017,The first stage at which the tick attaches its...,Mataki na farko da tsutsa ta haɗa kanta da jik...,B. egg,B. Kwaya
7,Animal Husbandry,2017,Which of the following organs is influenced by...,Wanne daga cikin waɗannan gabobin ne oxytocin ...,D. Udder,D. Udder
8,Animal Husbandry,2017,Which of the following methods are used in ani...,Wanne daga cikin waɗannan hanyoyin ana amfani ...,B. I and III only,B. I da III kawai
9,Animal Husbandry,2017,The vector that transmits trypanosomiasis is A...,Vektor da ke watsa trypanosomiasis shi ne Opti...,C. tsetse fly,C. Tsuntsin tsutsa


In [ ]:
print(df["question_number"].max())
print(df["question_number"].value_counts().sort_index().tail(20))

141
question_number
122     8
123     8
124     7
125    11
126     7
127     8
128     8
129     6
130    11
131     7
132     6
133     6
134     6
135     5
136     5
137     5
138     5
139     5
140     4
141     1
Name: count, dtype: int64


In [ ]:
# Fix mixed types in question_number
df["question_number"] = pd.to_numeric(df["question_number"], errors="coerce")  # converts "one" → NaN
df["question_number"] = df["question_number"].fillna(0).astype(int)

# Now push
from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset.push_to_hub("honourjesus/nllb-hausa-waec-translations")
print("✅ Pushed to Hugging Face!")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  44%|####3     |  526kB / 1.21MB            

✅ Pushed to Hugging Face!


In [ ]:
# Push as a dataset
from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset.push_to_hub("honourjesus/nllb-hausa-waec-translations")
print("✅ Pushed to Hugging Face!")

ArrowInvalid: ("Could not convert 'one' with type str: tried to convert to double", 'Conversion failed for column question_number with type object')